In [1]:
!pip install groq

In [2]:
import os
from google.colab import userdata

api_key = userdata.get("GROK_API_KEY")
os.environ["GROQ_API_KEY"] = api_key

In [4]:
import functools
import json

from google.colab import userdata
from groq import APIConnectionError, Groq, RateLimitError

client = Groq(api_key=userdata.get("GROK_API_KEY"))
MODEL = "openai/gpt-oss-20b"

# Sample prices in rupees per kilogram
MENU = {
    "tomatoes": 40,
    "onions": 35,
    "potatoes": 30,
    "carrots": 60,
    "cucumbers": 45,
    "green beans": 80,
}

SYSTEM = f"""
You are the customer support assistant for Fresh Basket Vegetable Shop.

FACTS:
- We sell fresh vegetables and take orders for in-store pickup.
- Prices in rupees per kilogram: {json.dumps(MENU, ensure_ascii=False)}
- Prices and stock can change daily.
- Pickup times must be confirmed by the shop.

RULES:
- Use the listed prices when giving an estimate. Say that the shop
  will confirm the final price and availability.
- Ask for the quantity and unit if either is unclear.
- When estimating a total, convert grams to kilograms before multiplying
  by the price per kilogram. For example, 500 g is 0.5 kg.
- Suggest vegetables based on the customer's recipe or budget.
- If an item is confirmed out of stock, suggest a suitable substitute.
- Only confirm an order or pickup time after the shop verifies it.
- Use details the customer has already shared in the conversation.
- Answer questions about vegetables, prices, pickup, and vegetable storage.

FORMAT:
- Reply politely in English or Telugu, matching the customer.
- Keep replies under 80 words when possible.
- Show prices clearly in rupees and use bullets for multiple items.
"""

def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"   [log] {func.__name__}{args} -> {result!r}")
        return result

    return wrapper


@log_call
def get_price(item: str, quantity_kg: float = 1) -> float:
    """Calculate the price of a vegetable for a weight in kilograms."""
    item = item.lower().strip()
    if item not in MENU:
        raise ValueError(f"No listed price for {item}.")
    if quantity_kg <= 0:
        raise ValueError("Quantity must be greater than zero.")
    return round(MENU[item] * quantity_kg, 2)


history = []
MAX_TURNS = 8


def chat(user_text: str) -> str:
    user_message = {"role": "user", "content": user_text}

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM},
                *history,
                user_message,
            ],
            temperature=0.3,
        )
    except RateLimitError:
        return "We're a little busy. Please try again in a minute."
    except APIConnectionError:
        return "I can't reach the server right now. Please try again shortly."

    reply = response.choices[0].message.content
    if not reply:
        return "I couldn't generate a reply. Please try again."

    history.extend([user_message, {"role": "assistant", "content": reply}])
    del history[:-2 * MAX_TURNS]
    return reply


print(f"Price for 500 g of carrots: Rs. {get_price('carrots', 0.5):.2f}")
print("\nWelcome to Fresh Basket Vegetable Shop! 🥕")
print("How can I help you today? Type 'exit' to stop.\n")

while True:
    text = input("You: ").strip()

    if text.lower() in ("bye", "exit", "quit"):
        print("Bot: Thank you for visiting Fresh Basket. Have a great day!")
        break

    if text:
        print("Bot:", chat(text))

   [log] get_price('carrots', 0.5) -> 30.0
Price for 500 g of carrots: Rs. 30.00

Welcome to Fresh Basket Vegetable Shop! 🥕
How can I help you today? Type 'exit' to stop.

You: I need 1 kg of tomatoes and 500 g of carrots. How much will it cost?
Bot: Sure!  
- 1 kg tomatoes: ₹40  
- 0.5 kg carrots: ₹30  

**Estimated total:** ₹70  

The shop will confirm the final price and availability before you pick up. Let me know if you’d like to add anything else or choose a pickup time.
You: What vegetables can I buy for ₹100?
Bot: Here are a few options that fit within ₹100 (prices per kg):

- 1 kg tomatoes + 0.5 kg carrots = ₹70  
- 1 kg onions + 1 kg potatoes = ₹65  
- 1 kg cucumbers + 0.5 kg green beans = ₹72.5  
- 1 kg carrots + 1 kg potatoes = ₹90  

Feel free to mix and match or let me know your preference!
You: Can I pick up 2 kg of potatoes this evening?
Bot: I’ll check the shop’s schedule for this evening.  
Could you let me know the exact time you’d like to pick up, or would you prefe